In [1]:
import torch
from IPython.display import display

from synthetic_data_generation import make_cohort
import pandas as pd

RANDOM_STATE = 0

# 0 Generate Synthetic Data

In [2]:
data_folder = '../../../data/synthetic_data'
make_cohort(data_folder)

In [3]:
tabular_df = pd.read_csv(data_folder + '/clinical.csv', index_col=0)
display(tabular_df.head())

,age,gender,stage
patient_id,,,
patient_000,63.047171,M,late
patient_001,49.600159,F,early
patient_002,NaN,F,early
patient_003,69.405647,M,early
patient_004,40.489648,F,early


# 1 Data Loading

## 1.1 Tabular

In [4]:
from kalecancer.loaddata import Tabular

clinical_df = tabular_df.drop(columns =['stage'])

clinical_modality = Tabular(clinical_df)

# proof it works!
display(clinical_modality.frame.head())
display(clinical_modality['patient_000'])

,age,gender
id,,
patient_000,63.047171,M
patient_001,49.600159,F
patient_002,NaN,F
patient_003,69.405647,M
patient_004,40.489648,F


,age,gender
id,,
patient_000,63.047171,M


## 1.2 Patch Features
Patch Features = Whole Slide Images passed through an encoder such as UNI

In [5]:
from kalecancer.loaddata import PatchFeatures

patch_features = PatchFeatures.from_glob(

    # 'what files are the patches stored in?'
    pattern = data_folder + '/patches/*.h5',

    # 'how to extract the ids from these files'
    id_pattern= r'(patient_\d{3})\.h5'
)

display(patch_features['patient_000'].shape)
display(patch_features['patient_000'].head())

(168, 35)

file      x  \
id          patch                                                             
patient_000 0      ../../../data/synthetic_data/patches/patient_0...  26803   
            1      ../../../data/synthetic_data/patches/patient_0...  25438   
            2      ../../../data/synthetic_data/patches/patient_0...  36345   
            3      ../../../data/synthetic_data/patches/patient_0...  35263   
            4      ../../../data/synthetic_data/patches/patient_0...  41517   

                       y  feature_0  feature_1  feature_2  feature_3  \
id          patch                                                      
patient_000 0      30774   3.152563   1.471492  -2.566658  -0.236850   
            1      16956   3.750869   1.820646   0.730775  -1.572040   
            2      43944   3.232878   0.234401   0.270343  -0.863345   
            3      38608   3.524188   0.276274  -1.412766  -2.310103   
            4      18368   4.353586   0.092127  -0.837398  -0.594400   

                   feature_4  feature_5  feature_6  ...  feature_22  \
id          patch                                   ...               
patient_000 0       0.176512   0.295994  -0.371915  ...    0.438993   
            1      -0.066953  -1.172007  -0.518280  ...   -2.063238   
            2      -0.147529  -0.152523   0.383394  ...   -2.003522   
            3       0.054354  -0.471776   0.459386  ...   -1.296472   
            4      -1.480536  -0.888134  -0.358017  ...    0.328662   

                   feature_23  feature_24  feature_25  feature_26  feature_27  \
id          patch                                                               
patient_000 0        0.827988   -0.296571    0.066546   -0.697424    0.989584   
            1       -0.591103    0.590906   -1.581594    1.475949    0.368357   
            2        1.604254   -0.457699    0.107880    1.309551   -1.602260   
            3       -1.482185   -2.333616   -0.678264    0.749434   -0.284884   
            4        0.361922    1.320662   -0.342786   -1.476858    1.067222   

                   feature_28  feature_29  feature_30  feature_31  
id          patch                                                  
patient_000 0       -1.178304    0.782350   -0.190651    1.171247  
            1        0.846584   -0.570944    0.813764    1.068472  
            2       -1.251647   -1.601278   -0.794136    0.439637  
            3        0.197790    1.089218    1.327686   -0.069138  
            4       -0.331488    1.114592    0.383377   -0.131138  

[5 rows x 35 columns]

## 1.3 The Target

In [6]:
from kalecancer.loaddata import Classification

target = Classification(labels= tabular_df['stage'], classes=['early', 'late'])
display(target[['patient_000', 'patient_003']])


,label
id,
patient_000,late
patient_003,early


## 1.4 Multimodal Dataset: All Modalities Combined

In [7]:
from kalecancer.loaddata import MultimodalDataset

dataset = MultimodalDataset(
    modalities={
       'clinical': clinical_modality,
       'patches': patch_features
    },
    target=target,
    required_modalities=['clinical', 'patches']
)

display(dataset.summary())
display(dataset['patient_052'].keys())
display(dataset['patient_052']['clinical'])

,count
available: clinical,100
available: patches,100
available: target,100
included,100
included with clinical,100
included with patches,100
label: early,53
label: late,47


dict_keys(['clinical', 'patches', 'target'])

,age,gender
id,,
patient_052,NaN,F


# 2 Model Building

## 2.1 Intermediate Fusion

In [8]:
import base64
from IPython.display import Image, display

graph = """
flowchart LR
    A[Modality A<br/>e.g. Clinical table] --> EA
    B[Modality B<br/>e.g. Pathology patches] --> EB
    subgraph M[The Model]
        EA[Encoder A]
        EB[Encoder B]
        F((Fusion))
        H[Shared prediction head]
        EA -->|embedding A| F
        EB -->|embedding B| F
        F -->|fused representation| H
    end
    H --> O([Prediction])
"""

encoded = base64.urlsafe_b64encode(graph.encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}"))

In [9]:
# Needed to ensure model weights are initialised consistently
torch.manual_seed(RANDOM_STATE)

from kalecancer.model import ABMIL, IntermediateFusion, Concat, ClassificationHead

inter_model = IntermediateFusion(
    encoders = {
        'clinical': [torch.nn.Linear(dataset.modalities['clinical'].frame.shape[1], 8)],
        'patches': [
            ABMIL(32, hidden_dim=16, attention_dim=8, dropout=0.0),
            torch.nn.Linear(16,8)
        ]
    },

    # the encoder outputs concatenated
    fusion = Concat(),

    head = ClassificationHead(in_dim=16, n_classes=2)
)

display(inter_model)

IntermediateFusion(
  (encoders): ModuleDict(
    (clinical): EncoderList(
      (0): Linear(in_features=2, out_features=8, bias=True)
    )
    (patches): EncoderList(
      (0): ABMIL(
        (embed): Sequential(
          (0): Linear(in_features=32, out_features=16, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.0, inplace=False)
        )
        (attention_value): Sequential(
          (0): Linear(in_features=16, out_features=8, bias=True)
          (1): Tanh()
        )
        (attention_gate): Sequential(
          (0): Linear(in_features=16, out_features=8, bias=True)
          (1): Sigmoid()
        )
        (attention_score): Linear(in_features=8, out_features=1, bias=True)
      )
      (1): Linear(in_features=16, out_features=8, bias=True)
    )
  )
  (fusion): Concat()
  (head): ClassificationHead(
    (linear): Linear(in_features=16, out_features=2, bias=True)
  )
)

# 3 The Model Orchestrator

## 3.1 Creating the Transformation
Trained/fitted transforms must be passed into the `Pipeline` object, and not applied beforehand.

Training/fitting transforms on the full dataset (train + test) is data leakage!

In [10]:
# from the above information we make must make the following transforms.
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

numeric_pipeline = make_pipeline(SimpleImputer(), StandardScaler())

tab_transfromers = ColumnTransformer([
    ('numeric', numeric_pipeline, ['age']),
    ('categorical', OrdinalEncoder(encoded_missing_value=-1), ['gender'])
])
tab_transfromers

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

## 3.2 Creating the Orchestrator: `Pipeline`

In [11]:
from kalecancer.pipeline import Pipeline
from functools import partial

pipeline = Pipeline(
    model= inter_model,

    # these are transforms that will be applied to the dataset
    # currently only Tabular modalities accept transforms
    # transforms need to ouptut numerical values, and their output should be
    # accepted by the first encoder
    transforms={'clinical': tab_transfromers},

    optimizer= partial(torch.optim.AdamW, lr=1e-3),
    batch_size=32,
    drop_last=True,
    max_epochs=40,
    validation=None,
    early_stopping=None,
    accelerator='cpu',
    precision='32-true',
    random_state=RANDOM_STATE,
)
pipeline

,model,IntermediateF...as=True) ) )
,transforms,{'clinical': ColumnTransfo... ['gender'])])}
,optimizer,"functools.par...W'>, lr=0.001)"
,batch_size,32
,drop_last,True
,max_epochs,40
,validation,None
,early_stopping,None
,accelerator,'cpu'
,precision,'32-true'
,random_state,0


# 4 Training

## 4.1 Fitting

In [12]:
from kalecancer.loaddata import train_test_split
train, test = train_test_split(dataset, test_size=0.2, stratify=True, random_state=RANDOM_STATE)

In [13]:
pipeline.fit(train)

,model,IntermediateF...as=True) ) )
,transforms,{'clinical': ColumnTransfo... ['gender'])])}
,optimizer,"functools.par...W'>, lr=0.001)"
,batch_size,32
,drop_last,True
,max_epochs,40
,validation,None
,early_stopping,None
,accelerator,'cpu'
,precision,'32-true'
,random_state,0


In [14]:
display(pipeline.history_.head())

,train_loss,optimizer_steps,skipped_batches
epoch,,,
0,0.693046,2,0
1,0.698229,4,0
2,0.708995,6,0
3,0.693900,8,0
4,0.701664,10,0


# 5 Evaluation
Note that we are using small synthetic illustrative cohort, so results don't provide real clinical insight

## 5.1 Prediction

In [15]:
pipeline.predict(test).head()

,probability[early],probability[late]
id,,
patient_001,0.552869,0.447131
patient_004,0.652484,0.347516
patient_005,0.573326,0.426674
patient_012,0.533786,0.466215
patient_013,0.431970,0.568030


## 5.2 Evaluation

In [16]:
from kalecancer.evaluate import AUROC, BalancedAccuracy
pipeline.evaluate(test, metrics={"auroc": AUROC(positive_class='late'), "balanced_acc": BalancedAccuracy()})

auroc           0.727273
balanced_acc    0.631313
Name: score, dtype: float64